# Crawl One Website (Bare Minimum Crawler)

In [23]:
import requests

url = 'https://cnn.com'
html = requests.get(url).text

In [24]:
html[0:500]

'  <!DOCTYPE html>\n<html lang="en" data-uri="cms.cnn.com/_pages/clg34ol9u000047nodabud1o2@published" data-layout-uri="cms.cnn.com/_layouts/layout-homepage/instances/homepage-domestic@published"  data-site="cnn">\n  <head>\n<link rel="preload" href="/fonts/cnn/cnn_sans_display-v1.woff2" as="font" type="font/woff2" crossorigin="anonymous">\n<link rel="preload" href="/fonts/cnn/cnn_sans_display-medium-v1.woff2" as="font" type="font/woff2" crossorigin="anonymous">\n<link rel="preload" href="/fonts/cnn/cn'

# Crawl One Website (Extended)

In [25]:
import requests
from bs4 import BeautifulSoup

In [32]:
url = 'https://cnn.com'

html = requests.get(url).text
soup = BeautifulSoup(html, "html.parser")

In [33]:
links = [a.get("href") for a in soup.find_all("a")]
links[0:10]

['https://www.cnn.com',
 'https://www.cnn.com/us',
 'https://www.cnn.com/world',
 'https://www.cnn.com/politics',
 'https://www.cnn.com/business',
 'https://www.cnn.com/health',
 'https://www.cnn.com/entertainment',
 'https://www.cnn.com/cnn-underscored',
 'https://www.cnn.com/style',
 'https://www.cnn.com/travel']

In [34]:
images = sorted(set([img.get("src") for img in soup.find_all("img") if img.get("src") is not None]))
images[0:10]

['/media/sites/cnn/app-store-cnn-app-qr-code.png',
 '/media/sites/cnn/google-play-cnn-app-qr-code.png',
 'https://media.cnn.com/api/v1/images/stellar/bleacherreport/20260901133526011-ap-steelers-lions-football-52432.png?c=16x9&q=h_438,w_780,c_fill',
 'https://media.cnn.com/api/v1/images/stellar/bleacherreport/20260901183621434-getty-anaheim-california-los-angeles-angels-owner-arte-moreno-in-attendance-for-an-opening-day-game.png?c=2x3&q=h_384,w_256,c_fill',
 'https://media.cnn.com/api/v1/images/stellar/bleacherreport/20260901233415519-ap-rams-donald-comeback-16587.png?c=2x3&q=h_384,w_256,c_fill',
 'https://media.cnn.com/api/v1/images/stellar/bleacherreport/2233171148-large-cropped.jpg?c=2x3&q=h_384,w_256,c_fill',
 'https://media.cnn.com/api/v1/images/stellar/bleacherreport/2284607937-large-cropped.jpg?c=2x3&q=h_384,w_256,c_fill',
 'https://media.cnn.com/api/v1/images/stellar/prod/008-wp4-temple-with-no-fence-obstruction.jpg?c=2x3&q=h_384,w_256,c_fill',
 'https://media.cnn.com/api/v1/im

# Pull Full URL Context

In [35]:
import re, json, requests, trafilatura, nltk
import pandas as pd

from collections import Counter
from nltk.tokenize import sent_tokenize
from scipy.stats import entropy

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

from tqdm import tqdm

# add/run these to prevent the nltk silent killer

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\itsgo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\itsgo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [36]:
def extract_web_text_context(url):

    response = requests.get(url, timeout=5)

    result = trafilatura.extract(response.text, output_format="json")

    if result is None:
        return None

    data = json.loads(result)

    domain = urlparse(url).netloc.replace("www.", "")

    soup = BeautifulSoup(response.text, "html.parser")

    page_title = None

    if soup.title and soup.title.string:
        page_title = soup.title.string.strip()

    data['page_title'] = page_title

    links = []

    for anchor in soup.find_all("a", href=True):
        links.append(urljoin(url, anchor["href"]))

    for iframe in soup.find_all("iframe", src=True):
        links.append(urljoin(url, iframe["src"]))

    links = [link.replace("www.", "") for link in links]
    links = sorted(set(links))

    images = []

    for img in soup.find_all("img", src=True):
        images.append(urljoin(url, img["src"]))

    images = sorted(set(images))

    linked_domains = [urlparse(link).netloc for link in links]
    linked_domains = sorted(set(linked_domains))

    data['links'] = links
    data['images'] = images
    data['domain'] = domain
    data['linked_domains'] = linked_domains

    text = data['text']

    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r" +", " ", text)
    text = text.strip()

    data['url'] = url
    data['text'] = text
    data['tokens'] = text.split()

    sentences = sent_tokenize(text)
    data['sentences'] = sentences

    # summary statistics
    data['token_count'] = len(data['tokens'])
    data['sentence_count'] = len(data['sentences'])
    data['lexical_diversity'] = len(set(data['tokens'])) / data['token_count']

    # Shannon's Entropy - Information Theory 
    counts = Counter(data['tokens'])
    data['entropy'] = float(entropy(list(counts.values()), base=2))

    return data

In [37]:
url = 'https://cnn.com'

data = extract_web_text_context(url)
data.keys()

dict_keys(['text', 'comments', 'page_title', 'links', 'images', 'domain', 'linked_domains', 'url', 'tokens', 'sentences', 'token_count', 'sentence_count', 'lexical_diversity', 'entropy'])

In [38]:
data['text'][0:1000]

'- • Live Updates• Live UpdatesLive Updates Warning signs were visible ahead of Nepal-China floods, report says - • Video 3:01Video 3:01Video CNN joins Nepali police on arduous search for flood victims 3:01 - • Video 5:22Video 5:22Video CNN speaks to Nepali girl who lost 17 family members in floods 5:22 More Top Stories - Tropical Storm Edouard makes landfall near Texas-Louisiana border - Porn addiction, sharing water, secret son: Catch up on the day’s stories - • Streaming NowStreaming NowStreaming Now Why the bond market going up affects you - Global bonds sell off as Middle East conflict escalates, further stoking inflation fears - • Video 1:41Video 1:41Video Can this unlikely pair reshape Israeli politics? 1:41 - Are you a recent high school graduate who found a job quickly? Crime and courts Ad Feedback Ad Feedback Across our coverage - ‘Grey Beard’: 91-year-old man becomes oldest hiker to complete the Appalachian Trail - Video Why is Copenhagen the world’s most livable city? 1:29 

In [39]:
data['page_title']

'Breaking News, Latest News and Videos | CNN'

In [40]:
data['links']

['https://arabic.cnn.com?hpt=header_edition-picker',
 'https://bleacherreport.com/',
 'https://bleacherreport.com/?utm_source=cnn.com&utm_medium=referral&utm_campaign=editorial',
 'https://bleacherreport.com/articles/25494981-updated-br-nfl1000-team-rankings-after-2026-roster-cuts?utm_source=cnn.com&utm_medium=referral&utm_campaign=editorial',
 'https://bleacherreport.com/articles/25495242-what-know-about-thea-frodin-serena-williams-former-movie-body-double-making-us-open-debut-17?utm_source=cnn.com&utm_medium=referral&utm_campaign=editorial',
 'https://bleacherreport.com/articles/25495288-stan-kroenke-buy-angels-reported-mlb-record-price-rams-owner-now-controls-6-pro-teams?utm_source=cnn.com&utm_medium=referral&utm_campaign=editorial',
 'https://bleacherreport.com/articles/25495377-caviar-nuggets-and-top-us-open-2026-concession-menu-items-prices-revealed-video?utm_source=cnn.com&utm_medium=referral&utm_campaign=editorial',
 'https://bleacherreport.com/articles/25495396-so-f-ked-nfl-gm

In [41]:
data['domain']

'cnn.com'

In [42]:
data['linked_domains']

['arabic.cnn.com',
 'bleacherreport.com',
 'careers.wbd.com',
 'cnn.com',
 'cnn.it',
 'cnn.onelink.me',
 'cnn10.com',
 'cnnespanol.cnn.com',
 'edition.cnn.com',
 'facebook.com',
 'help.cnn.com',
 'instagram.com',
 'linkedin.com',
 'threads.com',
 'tiktok.com',
 'twitter.com',
 'us.cnn.com']

In [43]:
data['url']

'https://cnn.com'

In [44]:
data['tokens'][0:10]

['-',
 '•',
 'Live',
 'Updates•',
 'Live',
 'UpdatesLive',
 'Updates',
 'Warning',
 'signs',
 'were']

In [45]:
data['sentences'][0:10]

['- • Live Updates• Live UpdatesLive Updates Warning signs were visible ahead of Nepal-China floods, report says - • Video 3:01Video 3:01Video CNN joins Nepali police on arduous search for flood victims 3:01 - • Video 5:22Video 5:22Video CNN speaks to Nepali girl who lost 17 family members in floods 5:22 More Top Stories - Tropical Storm Edouard makes landfall near Texas-Louisiana border - Porn addiction, sharing water, secret son: Catch up on the day’s stories - • Streaming NowStreaming NowStreaming Now Why the bond market going up affects you - Global bonds sell off as Middle East conflict escalates, further stoking inflation fears - • Video 1:41Video 1:41Video Can this unlikely pair reshape Israeli politics?',
 '1:41 - Are you a recent high school graduate who found a job quickly?',
 'Crime and courts Ad Feedback Ad Feedback Across our coverage - ‘Grey Beard’: 91-year-old man becomes oldest hiker to complete the Appalachian Trail - Video Why is Copenhagen the world’s most livable ci

In [46]:
data['token_count']

548

In [47]:
data['sentence_count']

6

In [48]:
data['lexical_diversity']

0.6532846715328468

In [49]:
data['entropy']

7.762702714020416

# Put the Data in a DataFrame

In [50]:
crawl_df = pd.DataFrame([data])
crawl_df

,text,comments,page_title,links,images,domain,linked_domains,url,tokens,sentences,token_count,sentence_count,lexical_diversity,entropy
0,- • Live Updates• Live UpdatesLive Updates War...,,"Breaking News, Latest News and Videos | CNN",[https://arabic.cnn.com?hpt=header_edition-pic...,[https://cnn.com/media/sites/cnn/app-store-cnn...,cnn.com,"[arabic.cnn.com, bleacherreport.com, careers.w...",https://cnn.com,"[-, •, Live, Updates•, Live, UpdatesLive, Upda...",[- • Live Updates• Live UpdatesLive Updates Wa...,548,6,0.653285,7.762703
